![BeliefLens](assets/belieflens-notebook-header.png)

# Apply a frozen BeliefLens calibration inside LangChain

## Goal

Show how a LangChain or LangGraph workflow can convert an already observed language-probability vector into calibrated probabilities over **Risk-on**, **Mixed**, and **Risk-off** without calling a hosted BeliefLens service.

## What happens

1. Load the frozen financial benchmark schema and JSON calibration map.
2. Select one archived language measurement.
3. Apply the declared additive-log-ratio transformation and frozen multinomial calibration locally.
4. Return calibrated state probabilities through a LangChain runnable.
5. Route the next workflow action using the resulting uncertainty decision.

This notebook makes **zero BeliefLens API calls and zero model-provider calls**. It does not refit calibration. The CPU work is a few logarithms and a small matrix multiplication. Code inputs are hidden by default; outputs remain visible.

**Further information:** [BeliefLens primer](https://belieflens.org/#/primer) · [Software and integration guide](https://belieflens.org/#/software) · [Benchmark Lab](https://belieflens.org/#/benchmarks)


## 1. Prepare a small LangChain kernel

Only `langchain-core` is needed. To avoid changing dependencies in a large shared environment, create a small kernel from this repository:

```bash
python3 -m venv .venv-belieflens-langchain
source .venv-belieflens-langchain/bin/activate
python -m pip install --upgrade pip
python -m pip install "langchain-core>=0.3,<1" ipykernel numpy pandas
python -m ipykernel install --user --name belieflens-langchain --display-name "BeliefLens + LangChain"
```

Then select **Kernel → Change Kernel → BeliefLens + LangChain**. No BeliefLens API key is required for this local example.


In [1]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from langchain_core.runnables import RunnableLambda

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = Path('examples/notebooks/finance')
BUNDLE = ROOT / 'data' / 'offline_reproduction'
assert BUNDLE.exists(), f'Frozen finance bundle not found: {BUNDLE.resolve()}'
print('Local LangChain calibration environment is ready.')


Local LangChain calibration environment is ready.


## 2. Load the frozen measurement profile

The JSON file is the fitted measurement channel. It declares the response candidates, the additive-log-ratio transformation, the state ordering, regularization setting, coefficients, intercept and calibration-record identifiers. Loading it applies an existing calibration; it does not estimate a new one.


In [2]:
profile_path = BUNDLE / 'semantic_map.json'
profile = json.loads(profile_path.read_text())
profile_hash = hashlib.sha256(profile_path.read_bytes()).hexdigest()

pd.Series({
    'schema': profile['schema_version'],
    'states': ', '.join(profile['states']),
    'input candidates': ', '.join(profile['input_candidates']),
    'transformation': profile['input_transform'],
    'calibration records': len(profile['calibration_scenario_ids']),
    'profile SHA-256': profile_hash,
}, name='Frozen local measurement profile')


schema                                 belieflens-frozen-semantic-map-v1
states                                          Risk-on, Mixed, Risk-off
input candidates                        Risk-on, Mixed, Risk-off, Unsure
transformation                         additive log ratio against Unsure
calibration records                                                  180
profile SHA-256        ff93a18afd9362aa1f14746c797e0cba2fb0076c2c5cab...
Name: Frozen local measurement profile, dtype: object

## 3. Define the local calibrated measurement

The runnable expects probabilities over the four controlled language responses: Risk-on, Mixed, Risk-off and Unsure. It forms three log ratios against Unsure, applies the frozen coefficient matrix and intercept, and normalizes the resulting scores with the softmax function. The output is a probability distribution over the three declared financial states.

In a live workflow, an upstream provider-specific node would supply the phrase probabilities. Here we use an archived vector so the example remains deterministic, free and auditable.


In [3]:
states = profile['states']
candidates = profile['input_candidates']
coefficients = np.asarray(profile['coefficients'], dtype=float)
intercept = np.asarray(profile['intercept'], dtype=float)

def apply_frozen_calibration(value):
    supplied = value['language_probabilities']
    probabilities = np.asarray([float(supplied[name]) for name in candidates], dtype=float)
    if np.any(probabilities <= 0) or not np.isclose(probabilities.sum(), 1.0, atol=1e-6):
        raise ValueError('Language probabilities must be positive and sum to one.')

    alr = np.log(probabilities[:3] / probabilities[3])
    scores = coefficients @ alr + intercept
    calibrated = np.exp(scores - scores.max())
    calibrated /= calibrated.sum()
    state_probabilities = dict(zip(states, calibrated.tolist()))
    leading_state = max(state_probabilities, key=state_probabilities.get)
    leading_probability = state_probabilities[leading_state]
    decision = 'measurement_accepted_within_scope' if leading_probability >= 0.80 else 'measurement_valid_state_ambiguous'
    return {
        **value,
        'state_probabilities': state_probabilities,
        'leading_state': leading_state,
        'decision': decision,
        'measurement_profile_sha256': profile_hash,
        'execution_mode': 'local_frozen_calibration',
        'provider_calls': 0,
        'belieflens_api_calls': 0,
    }

measurement = RunnableLambda(apply_frozen_calibration)
print('Local calibrated measurement runnable created.')


Local calibrated measurement runnable created.


## 4. Invoke the measurement as a LangChain node

The archived row supplies the observable language probabilities and the previously stored calibrated probabilities. The latter are used only as a reproduction check: the runnable must recover them from the JSON map to numerical precision.


In [4]:
rows = pd.read_csv(BUNDLE / 'row_level_measurements.csv')
archived = rows.iloc[0]
record = {
    'observation_id': archived['scenario_id'],
    'evidence_date': archived['date'],
    'language_probabilities': {
        'Risk-on': archived['raw_risk_on'],
        'Mixed': archived['raw_mixed'],
        'Risk-off': archived['raw_risk_off'],
        'Unsure': archived['raw_unsure'],
    },
}
result = measurement.invoke(record)

archived_calibrated = np.asarray([
    archived['calibrated_risk_on'], archived['calibrated_mixed'], archived['calibrated_risk_off']
])
locally_calibrated = np.asarray([result['state_probabilities'][state] for state in states])
reproduction_error = float(np.max(np.abs(archived_calibrated - locally_calibrated)))

print(json.dumps({
    'observation_id': result['observation_id'],
    'language_probabilities': result['language_probabilities'],
    'calibrated_state_probabilities': result['state_probabilities'],
    'leading_state': result['leading_state'],
    'decision': result['decision'],
    'maximum_reproduction_error': reproduction_error,
    'BeliefLens API calls': result['belieflens_api_calls'],
    'provider calls': result['provider_calls'],
}, indent=2))
assert reproduction_error < 1e-12


{
  "observation_id": "daily-2018-01-10",
  "language_probabilities": {
    "Risk-on": 0.999999999997,
    "Mixed": 1.0000228884425302e-12,
    "Risk-off": 1.0000228884425302e-12,
    "Unsure": 1.0000228884425302e-12
  },
  "calibrated_state_probabilities": {
    "Risk-on": 0.6308426961449894,
    "Mixed": 0.3453985594919225,
    "Risk-off": 0.02375874436308812
  },
  "leading_state": "Risk-on",
  "decision": "measurement_valid_state_ambiguous",
  "maximum_reproduction_error": 2.0816681711721685e-17,
  "BeliefLens API calls": 0,
  "provider calls": 0
}


## 5. Gate the next LangGraph action

The calibrated distribution—not a printed confidence number—becomes part of the workflow state. A conditional edge can continue, request review or abstain before a trade or other consequential tool call. The threshold below illustrates orchestration; it is not an investment-policy recommendation.


In [5]:
def route_measurement(state):
    decision = state['measurement']['decision']
    if decision == 'measurement_accepted_within_scope':
        return 'continue'
    if 'abstain' in decision:
        return 'abstain'
    return 'human_review'

route = route_measurement({'measurement': result})
print('Next workflow route:', route)

# graph.add_conditional_edges('measure', route_measurement, {
#     'continue': 'action', 'human_review': 'review', 'abstain': 'stop'
# })


Next workflow route: human_review


## What this integration provides

LangChain supplies orchestration: it carries the probability vector into the measurement node and routes the resulting state. The frozen BeliefLens JSON supplies the statistical measurement: declared states, transformation, fitted calibration and provenance hash. Because this notebook applies that profile locally, it creates no hosted workload and requires no BeliefLens credentials.

A production deployment may optionally replace the local runnable with a managed BeliefLens endpoint to obtain centrally governed profiles, access control, shared audit logs and server-issued certificates. That is an explicit deployment choice, not a requirement for this reproducible example.
